In [1]:
import pandas as pd
print("=== Chart 4: Scrollytelling Dot Map Preprocessing ===")

# 1. Load the file
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Extract the year from the WEEK column for filtering
# We only want data from 2017 to 2025 to keep the story contemporary
df_me['YEAR'] = pd.to_datetime(df_me['WEEK']).dt.year
df_recent = df_me[(df_me['YEAR'] >= 2017) & (df_me['YEAR'] <= 2025)].copy()

# grouping
# We sum up all the events that occurred at the exact same geographical longitude and latitude
df_dotmap = df_recent.groupby(
    ['COUNTRY', 'ADMIN1', 'CENTROID_LATITUDE', 'CENTROID_LONGITUDE', 'EVENT_TYPE']
).agg({
    'EVENTS': 'sum',
    'FATALITIES': 'sum'
}).reset_index()

# 4. Filter events related to the story
event_types_to_keep = ['Protests', 'Riots', 'Violence against civilians', 'Battles']
df_dotmap = df_dotmap[df_dotmap['EVENT_TYPE'].isin(event_types_to_keep)]

# 5. Rename columns
df_dotmap.rename(columns={
    'COUNTRY': 'country',
    'ADMIN1': 'admin1',
    'CENTROID_LATITUDE': 'lat',
    'CENTROID_LONGITUDE': 'lon',
    'EVENT_TYPE': 'type',
    'EVENTS': 'events',
    'FATALITIES': 'fatalities'
}, inplace=True)

# 6. Save as a CSV file
output_file_4 = 'chart4_dotmap.csv'
df_dotmap.to_csv(output_file_4, index=False)

print(f"\n Number of new rows: {len(df_dotmap)}")
print(df_dotmap.head()[[ 'country', 'type', 'lat', 'lon', 'events' ]])
print(f"\nData saved to: {output_file_4}")

=== Chart 4: Scrollytelling Dot Map Preprocessing ===

 Number of new rows: 768
   country                        type      lat      lon  events
1  Bahrain                    Protests  26.1927  50.5508    3564
2  Bahrain                       Riots  26.1927  50.5508    1537
4  Bahrain  Violence against civilians  26.1927  50.5508      10
5  Bahrain                     Battles  26.2547  50.6428       1
7  Bahrain                    Protests  26.2547  50.6428     269

Data saved to: chart4_dotmap.csv


In [2]:
import pandas as pd

print("\n=== Chart 5: Slope Chart Data Preprocessing ===")

# 1. Load the Middle East file
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Extract the year
df_me['YEAR'] = pd.to_datetime(df_me['WEEK']).dt.year

# 3. Select target countries and two specific years for comparison
# We select 2017 and 2025, which have complete data
target_countries = ['Iran', 'Yemen', 'Syria', 'Iraq', 'Lebanon']
df_slope_base = df_me[df_me['COUNTRY'].isin(target_countries) & (df_me['YEAR'].isin([2017, 2025]))].copy()

# 4. Aggregate the total fatalities for each country in that specific year
df_slope = df_slope_base.groupby(['COUNTRY', 'YEAR'])['FATALITIES'].sum().reset_index()

# 5.Pivot (rotating the table)
# We need the year 2017 to be one column and the year 2025 to be another column. Pivot does this.
df_slope_pivot = df_slope.pivot(index='COUNTRY', columns='YEAR', values='FATALITIES').reset_index()

# Clean up the column names
df_slope_pivot.columns = ['country', 'year_2017', 'year_2025']

# 6. Save the cleaned file
output_file_5 = 'chart5_slope.csv'
df_slope_pivot.to_csv(output_file_5, index=False)

print("\n Chart 5 Data Saved:")
print(df_slope_pivot)
print(f"\nData saved to: {output_file_5}")


=== Chart 5: Slope Chart Data Preprocessing ===

 Chart 5 Data Saved:
   country  year_2017  year_2025
0     Iran        199       1009
1     Iraq      30546        597
2  Lebanon        389        411
3    Syria      54334       7847
4    Yemen      17559       2893

Data saved to: chart5_slope.csv


In [3]:
import pandas as pd

print("=== Chart 6: 100% Stacked Bar Preprocessing ===")

# 1. Load the Middle East data
file_name = 'Middle-East_aggregated_data_up_to-2026-01-24.xlsx'
df_me = pd.read_excel(file_name)

# 2. Filter recent years (2017 to 2025)
df_me['YEAR'] = pd.to_datetime(df_me['WEEK']).dt.year
df_recent = df_me[(df_me['YEAR'] >= 2017) & (df_me['YEAR'] <= 2025)].copy()

# 3. Select a few countries with different patterns for comparison
# Syria and Yemen (classic war pattern) vs. Iran and Bahrain (internal repression pattern)
comparison_countries = ['Syria', 'Yemen', 'Iran', 'Bahrain']
df_stacked = df_recent[df_recent['COUNTRY'].isin(comparison_countries)].copy()

# 4. Categorize events into the 3 main groups of our story
def categorize_event(event):
    if event in ['Battles', 'Explosions/Remote violence']:
        return 'Military Conflict'      # Military war/conflict
    elif event in ['Protests', 'Riots']:
        return 'Civil Unrest'           # Popular protests/Civil unrest
    elif event == 'Violence against civilians':
        return 'State Repression'       # One-sided violence and repression
    else:
        return 'Other'

df_stacked['Category'] = df_stacked['EVENT_TYPE'].apply(categorize_event)

# Remove the 'Other' rows
df_stacked = df_stacked[df_stacked['Category'] != 'Other']

# 5. Calculate the total number of events for each category in each country
df_group = df_stacked.groupby(['COUNTRY', 'Category'])['EVENTS'].sum().reset_index()

# 6. Pivot the table so that categories become columns
df_pivot = df_group.pivot(index='COUNTRY', columns='Category', values='EVENTS').fillna(0).reset_index()

# 7. Calculate percentages (so the columns become 100% Stacked)
df_pivot['Total_Events'] = df_pivot['Military Conflict'] + df_pivot['Civil Unrest'] + df_pivot['State Repression']

df_pivot['Military_pct'] = (df_pivot['Military Conflict'] / df_pivot['Total_Events']) * 100
df_pivot['Civil_pct'] = (df_pivot['Civil Unrest'] / df_pivot['Total_Events']) * 100
df_pivot['Repression_pct'] = (df_pivot['State Repression'] / df_pivot['Total_Events']) * 100

# 8. Save the file
output_file_6 = 'chart6_stacked.csv'
df_pivot.to_csv(output_file_6, index=False)

print("\n Chart 6 Data Ready:")
print(df_pivot[['COUNTRY', 'Military_pct', 'Civil_pct', 'Repression_pct']])

=== Chart 6: 100% Stacked Bar Preprocessing ===

 Chart 6 Data Ready:
Category  COUNTRY  Military_pct  Civil_pct  Repression_pct
0         Bahrain      0.171103  99.249049        0.579848
1            Iran      3.194867  94.423563        2.381570
2           Syria     83.290853   4.955995       11.753152
3           Yemen     69.039655  25.005155        5.955190
